# Interactive alignment/syntax plot — mar5

Run the cells below. Use the dropdowns to select **run**, **ablation type**, and **metric**, then click **Update Plot**.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

RUNS_DIR = Path("/n/netscratch/dam_lab/Lab/drooryck/codeswitching-llms/feb_exp/results/mar5/runs")
MAX_EXAMPLES_IN_HOVER = 6
MAX_EXAMPLE_LEN = 60


def load_metrics_df(runs_dir, ablation_type, run_filter=None):
    """Load ablation metrics JSONs into a DataFrame."""
    results = []
    for run_dir in sorted(runs_dir.glob("p*_run*")):
        parts = run_dir.name.split("_run")
        prop = float(parts[0][1:]) / 100.0
        run_id = int(parts[1])
        if run_filter is not None and run_id not in run_filter:
            continue
        json_path = run_dir / f"ablation_{ablation_type}_metrics.json"
        if not json_path.exists():
            continue
        with open(json_path) as f:
            metrics = json.load(f)
        results.append({"prop": prop, "run_id": run_id, "ablation": ablation_type, **metrics})
    if not results:
        raise FileNotFoundError(f"No ablation_{ablation_type}_metrics.json found in {runs_dir}")
    return pd.DataFrame(results).sort_values("prop")


def build_examples_lookup(runs_dir, ablation_type, run_filter=None, max_per_regime=10):
    """Build (prop, run_id, lang) -> list of example dicts from ablation_predictions.csv."""
    from collections import defaultdict
    lookup = defaultdict(list)
    for run_dir in sorted(runs_dir.glob("p*_run*")):
        parts = run_dir.name.split("_run")
        prop = float(parts[0][1:]) / 100.0
        run_id = int(parts[1])
        if run_filter is not None and run_id not in run_filter:
            continue
        csv_path = run_dir / "ablation_predictions.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        if "ablation" not in df.columns:
            continue
        sub = df[df["ablation"] == ablation_type]
        for lang in ("fr", "nl"):
            lang_sub = sub[sub["language"] == lang]
            key = (prop, run_id, lang)
            for _, row in lang_sub.head(max_per_regime).iterrows():
                lookup[key].append({
                    "input": str(row.get("input", "")),
                    "gold": str(row.get("gold", "")),
                    "prediction": str(row.get("prediction", "")),
                })
    return dict(lookup)


def format_examples(examples, max_len=MAX_EXAMPLE_LEN):
    if not examples:
        return "No examples."
    lines = []
    for i, ex in enumerate(examples[:MAX_EXAMPLES_IN_HOVER], 1):
        inp = ex["input"][:max_len] + ("…" if len(ex["input"]) > max_len else "")
        gold = ex["gold"][:max_len] + ("…" if len(ex["gold"]) > max_len else "")
        pred = ex["prediction"][:max_len] + ("…" if len(ex["prediction"]) > max_len else "")
        lines.append(f"{i}. {inp} → <b>{pred}</b>  (gold: {gold})")
    return "<br>".join(lines)


def build_figure(ablation, metric, run_ids):
    run_set = set(run_ids)
    df = load_metrics_df(RUNS_DIR, ablation, run_filter=run_set)
    examples = build_examples_lookup(RUNS_DIR, ablation, run_filter=run_set)

    fr_col = f"fr_{metric}_score"
    nl_col = f"nl_{metric}_score"
    if fr_col not in df.columns:
        raise ValueError(f"Column {fr_col} not found. Available: {sorted(df.columns)}")

    score_label = {"syntax": "Syntax score", "morphology": "Morphology score", "alignment": "Alignment score"}[metric]
    ylabel = {"syntax": "Syntax (0=NL, 1=FR)", "morphology": "Morphology (0=NL, 1=FR)", "alignment": "Alignment (0=NL, 1=FR)"}[metric]
    run_str = ", ".join(str(r) for r in sorted(run_ids))
    title = f"{score_label} — {ablation} ablation — run(s) {run_str}"

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=("French test sentences", "Dutch test sentences"),
                        horizontal_spacing=0.12)

    for panel_col, (lang, lang_col, color) in enumerate([
        ("fr", fr_col, "rgba(31,119,180,0.7)"),
        ("nl", nl_col, "rgba(255,127,14,0.7)"),
    ], start=1):
        for rid in sorted(run_set):
            rdf = df[df["run_id"] == rid].sort_values("prop")
            if rdf.empty:
                continue
            fig.add_trace(go.Scatter(
                x=rdf["prop"], y=rdf[lang_col],
                mode="lines", line=dict(color="gray", width=1),
                opacity=0.5, showlegend=False, hoverinfo="skip",
            ), row=1, col=panel_col)

        hover = []
        for _, row in df.iterrows():
            exs = format_examples(examples.get((row["prop"], row["run_id"], lang), []))
            hover.append(
                f"<b>Prop</b> {row['prop']:.2f} | <b>Run</b> {row['run_id']}<br>"
                f"<b>{score_label}</b> {row[lang_col]:.4f}<br><br>"
                f"<b>Examples ({lang.upper()}):</b><br>{exs}"
            )
        fig.add_trace(go.Scatter(
            x=df["prop"], y=df[lang_col],
            mode="markers", marker=dict(size=8, color=color, line=dict(width=0.5, color="white")),
            text=hover, hoverinfo="text", showlegend=False,
        ), row=1, col=panel_col)

        medians = df.groupby("prop")[lang_col].median()
        fig.add_trace(go.Scatter(
            x=medians.index, y=medians.values,
            mode="lines+markers",
            line=dict(color="black", width=2.5, dash="dash"),
            marker=dict(size=10, color="black"),
            name="Median" if panel_col == 1 else None,
            showlegend=(panel_col == 1), hoverinfo="skip",
        ), row=1, col=panel_col)

        fig.update_yaxes(title_text=ylabel, range=[0, 1], row=1, col=panel_col)
        fig.update_xaxes(title_text="Proportion FR in training", row=1, col=panel_col)

    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center"),
        template="plotly_white", width=1200, height=520,
        margin=dict(t=80, b=60, l=60, r=40),
    )
    return fig


# --- Widgets ---
run_select = widgets.SelectMultiple(
    options=[1, 2, 3], value=(1, 2, 3),
    description="Runs:", rows=3,
)
ablation_select = widgets.Dropdown(
    options=["none", "subject", "verb", "object"], value="subject",
    description="Ablation:",
)
metric_select = widgets.Dropdown(
    options=["alignment", "syntax", "morphology"], value="alignment",
    description="Metric:",
)
update_btn = widgets.Button(description="Update Plot", button_style="primary")
output = widgets.Output()

def on_update(_):
    with output:
        clear_output(wait=True)
        try:
            fig = build_figure(ablation_select.value, metric_select.value, list(run_select.value))
            fig.show()
        except Exception as e:
            print(f"Error: {e}")

update_btn.on_click(on_update)
display(widgets.HBox([run_select, ablation_select, metric_select, update_btn]))
display(output)

# Show initial plot
on_update(None)

Output()